# Trend-aware retrieval: "laptop twee"
"laptop twee" is an aesthetic, not laptops. Retrieval is split into five channels scored separately and fused late
(see `trend_search.py`): **board** (fuzzy board relevance, noisy-OR over all of a photo's boards), **visual**
(query-by-example from trend seeds), **ppr** (personalized PageRank over boards + visual kNN), **caption** (BM25 + bge),
**siglip** (direct text→image fallback).

In [ ]:
import os
os.environ.pop("PYTHONPATH", None)

import sys
sys.path = [
    p for p in sys.path
    if "/opt/software-current/" not in p
]

import regex
print("regex:", regex.__file__)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../../data/unsplash/clip_train.parquet")
collections_df = pd.read_csv("../../data/unsplash/collections.tsv000", sep="\t")
collections_df["collection_id"] = collections_df["collection_id"].astype(str)  # injected ids are strings

board_titles = (
    collections_df
    .groupby("photo_id")["collection_title"]
    .apply(lambda x: ", ".join(x.dropna().astype(str).unique()))
    .rename("collection_title")
)
df = df.merge(board_titles, on="photo_id", how="left")   # display only - never embedded
df["image_path"] = [f"../../data/unsplash/{i}" for i in df["image_path"]]

# untouched copies for the evaluation configs below
df_base, collections_base = df.copy(), collections_df.copy()

## Inject the "laptop twee" board
Fixes: (1) all 10 rows are added with a single concat, (2) the injected photos also get rows in `collections_df`
under one new `collection_id` so they have graph edges, and the photo/collection lookups are rebuilt afterwards.

In [ ]:
inject_df = pd.read_csv("../../data/small/metadata.csv")
inj = inject_df.iloc[:10].reset_index(drop=True)

new_rows = pd.DataFrame({
    "photo_id": [f"laptop_twee_{i}" for i in range(len(inj))],
    "image_path": [f"../../data/small/{p}" for p in inj["image_path"]],
    "caption": inj["caption"].astype(str).tolist(),
    "collection_title": "laptop twee",
})
df = pd.concat([df_base, new_rows], ignore_index=True)

collections_df = pd.concat([
    collections_base,
    pd.DataFrame({"photo_id": new_rows["photo_id"], "collection_id": "laptop_twee",
                  "collection_title": "laptop twee"}),
], ignore_index=True)

# rebuild lookups after injection
photo_to_idx = {p: i for i, p in enumerate(df["photo_id"])}
unique_collections = (collections_df[["collection_id", "collection_title"]]
                      .drop_duplicates("collection_id").reset_index(drop=True))
collection_to_idx = {c: i for i, c in enumerate(unique_collections["collection_id"])}

assert df["photo_id"].str.startswith("laptop_twee_").sum() == 10
assert (collections_df["collection_id"] == "laptop_twee").sum() == 10
print(len(df), "photos,", len(unique_collections), "collections")

## Models
The SigLIP helpers now live in `trend_search.py` (same signatures; `.pooler_output` vs plain-tensor outputs handled; text padded to `max_length=64`). The notebook's model is shared with the module.

In [ ]:
import torch
from transformers import AutoModel, AutoProcessor

import trend_search as ts
from trend_search import embed_images_batch, embed_text_batch

MODEL_NAME = "google/siglip2-base-patch16-224"
device = ts.get_device()
embed_model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
embed_processor = AutoProcessor.from_pretrained(MODEL_NAME)
ts.set_siglip(embed_model, embed_processor, device)

## Build the index
SigLIP image embeddings, sparse photo × collection matrix `A` (replaces the all-pairs edge list), visual kNN graph (k=10,
GPU `topk` in chunks), bge embeddings of each **unique normalized board title** (never the comma-joined string),
caption bge + BM25. Embeddings are cached per image path / text in `trend_cache/`, so later configs only embed what's new.

In [ ]:
index = ts.build_index(df, collections_df, cache_dir="trend_cache")
index.A.shape, index.A.nnz

### Calibrate board relevance
`r_c = sigmoid(((0.6·dense + 0.4·lex)·gate(cov) − t) / τ)`. Check that the top-50 boards look right and adjust `index.board_params.t` / `.tau` if not (then re-run). `cov` is the IDF-weighted share of query tokens a title covers; it keeps "Laptops" from matching "laptop twee".

In [ ]:
# index.board_params.t, index.board_params.tau = 0.62, 0.05
ts.print_board_calibration(index, ["laptop twee", "car", "cozy", "office siren", "dark academia"], top=50)

In [ ]:
groups = index.trend_groups("laptop twee", top_n=200, distance=0.15)
print("laptop-twee trend group:", groups.loc[groups.in_query_group, "title_norm"].tolist())
groups.head(20)

## Search and explain

In [ ]:
res = ts.search("laptop twee")
print({k: v for k, v in res.attrs.items() if k != "seed_ids"})
cols = ["photo_id", "caption", "s_board", "s_visual", "s_ppr", "s_caption", "s_siglip", "score", "top_channel", "is_seed"]
res[cols].head(25)

In [ ]:
# which boards drove the board score, per photo
index.board_breakdown("laptop twee").head(30)

In [ ]:
pid = res.loc[~res.is_seed, "photo_id"].iloc[0]   # best non-seed result
e = ts.explain(pid, "laptop twee")
print(pid, "rank", e["rank"], "| top channel:", e["top_channel"], "|", e["caption"])
display(e["channels"]); display(e["boards"])

## 5×5 grids (label = rank, top channel, fused score)

In [ ]:
import matplotlib.pyplot as plt

injected = new_rows["photo_id"].tolist()
for q, kw in [("laptop twee", {}), ("laptop twee", {"deliteralize": "text"}),
              ("laptop twee", {"weights": {"siglip": 1}}), ("car", {})]:
    r = ts.search(q, **kw)
    ts.plot_grid(r, title=f"{q} {kw or ''}", highlight_ids=injected)
    plt.show()

In [ ]:
# query-by-example only: the 10 injected images as explicit examples, no board needed
r = ts.search("laptop twee", example_images=new_rows["image_path"].tolist())
ts.plot_grid(r[~r.is_seed], title="laptop twee + example images (seeds hidden)"); plt.show()

## Evaluation
### 1. Held-out trend test
5 laptop-twee images in the board, the other 5 in `df` with **no** board membership. The in-board images and all seeds are
removed from every ranking, so each method ranks the same candidates. Ranks are pessimistic on ties.

In [ ]:
df_h, coll_h = ts.inject_trend(df_base, collections_base, inject_df, "../../data/small",
                               boards={"laptop_twee": ("laptop twee", range(5))})
heldout_ids = [f"laptop_twee_{i}" for i in range(5, 10)]
inboard_ids = [f"laptop_twee_{i}" for i in range(5)]
assert set(heldout_ids).isdisjoint(coll_h["photo_id"])

index_h = ts.build_index(df_h, coll_h, cache_dir="trend_cache")
heldout = ts.evaluate_heldout(index_h, "laptop twee", heldout_ids, inboard_ids)
heldout

In [ ]:
r = ts.search("laptop twee", index=index_h, exclude_ids=inboard_ids + index_h.search("laptop twee").attrs["seed_ids"])
ts.plot_grid(r, title="held-out test: fused (green = held-out trend images)", highlight_ids=heldout_ids); plt.show()

### 2. Fuzzy-title test
The injected board is split into differently-written variants; each must still match (`r ≥ 0.5`) and land in the trend group.

In [ ]:
variants = {"lt_a": ("Laptop-Twee 🌸", [0, 1, 2, 3]),
            "lt_b": ("twee laptop aesthetic", [4, 5, 6]),
            "lt_c": ("laptoptwee", [7, 8, 9])}
df_f, coll_f = ts.inject_trend(df_base, collections_base, inject_df, "../../data/small", boards=variants)
index_f = ts.build_index(df_f, coll_f, cache_dir="trend_cache")

fuzzy_boards, fuzzy = ts.evaluate_fuzzy(index_f, "laptop twee", {k: v[0] for k, v in variants.items()},
                                        [f"laptop_twee_{i}" for i in range(10)])
display(fuzzy_boards.round(3)); fuzzy

### 3. Regression: "car"
Weak label = caption matches `\bcars?\b` (case-insensitive; the old `str.contains("car")` also hit "card", "carpet"). The caption channel reads the same captions as the label, so **fused − caption** is the leakage-free comparison.

In [ ]:
regression = ts.evaluate_regression(index, "car", r"\bcars?\b", k=25)
regression

## Summary

In [ ]:
summary = pd.concat([heldout, fuzzy, regression], ignore_index=True)
metric_cols = [c for c in summary.columns if c.startswith(("recall@", "precision@", "median_rank"))]
with pd.option_context("display.max_columns", None, "display.width", 200):
    print(summary[["test", "method", "ranks", *metric_cols, "variants_matched", "not_worse_than_siglip"]]
          .round(3).fillna("").to_string(index=False))